# GitHub Repos — Visualization & Fun Facts

This notebook picks up after the cleaning step. It's all charts, comparisons, and easy math — no new pipelines, just exploring the data we already cleaned.

## Setup
Same cleaning as before, just so this notebook can run on its own.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv('ml_repositories.csv')

data['created_at'] = pd.to_datetime(data['created_at']).dt.tz_localize(None)
data['updated_at'] = pd.to_datetime(data['updated_at']).dt.tz_localize(None)
data['id'] = data['id'].astype(str)
data['language'] = data['language'].astype(str)

current_date = pd.to_datetime("2026-08-23")
data['repo_age_days'] = (current_date - data['created_at']).dt.days

data.head()

## 1. Top 10 repos by stars
Which project is the most popular?

In [ ]:
top10 = data.sort_values('stars', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top10['name'], top10['stars'], color='skyblue')
plt.xlabel('Stars')
plt.title('Top 10 Repos by Stars')
plt.gca().invert_yaxis()
plt.show()

## 2. What languages show up most?
A pie chart of every repo's language.

In [ ]:
language_counts = data['language'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(language_counts, labels=language_counts.index, autopct='%1.1f%%')
plt.title('Repos by Language')
plt.show()

## 3. Stars vs. forks for the top 5
Do people star a project more than they copy (fork) it?

In [ ]:
top5 = data.sort_values('stars', ascending=False).head(5)

x = range(len(top5))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - width/2 for i in x], top5['stars'], width=width, label='Stars')
plt.bar([i + width/2 for i in x], top5['forks'], width=width, label='Forks')
plt.xticks(list(x), top5['name'], rotation=30, ha='right')
plt.ylabel('Count')
plt.title('Stars vs Forks — Top 5 Repos')
plt.legend()
plt.tight_layout()
plt.show()

## 4. More charts to explore

**How many stars do repos usually have?** (a histogram — most repos will be low, a few will be huge)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data['stars'], bins=10, color='coral', edgecolor='black')
plt.xlabel('Stars')
plt.ylabel('Number of Repos')
plt.title('How Many Stars Do Repos Usually Have?')
plt.show()

**Do more stars mean more forks — for every repo, not just the top 5?**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(data['stars'], data['forks'], color='seagreen')
plt.xlabel('Stars')
plt.ylabel('Forks')
plt.title('Do More Stars Mean More Forks?')
plt.show()

**Which owner has the most repos in this list?**

In [ ]:
owner_counts = data['owner'].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(owner_counts.index, owner_counts.values, color='mediumpurple')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of Repos')
plt.title('Repos per Owner')
plt.tight_layout()
plt.show()

**Which repos have the most open issues?** (the "busiest" projects)

In [ ]:
top_issues = data.sort_values('open_issues', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_issues['name'], top_issues['open_issues'], color='tomato')
plt.xlabel('Open Issues')
plt.title('Which Repos Have the Most Open Issues?')
plt.gca().invert_yaxis()
plt.show()

**Does when a repo was created affect how many stars it has?** (a timeline)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(data['created_at'], data['stars'], color='steelblue')
plt.xlabel('Created Date')
plt.ylabel('Stars')
plt.title('Repo Creation Date vs Stars')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Bonus — bubble chart:** stars vs forks, where bigger bubbles mean more open issues.

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(data['stars'], data['forks'], s=data['open_issues'] * 3, alpha=0.5, color='orange', edgecolor='black')
plt.xlabel('Stars')
plt.ylabel('Forks')
plt.title('Stars vs Forks (bubble size = open issues)')
plt.show()

## 5. Detective questions
No charts needed — just reading the table closely.

**Which owner shows up more than once?**

In [ ]:
repeat_owners = data['owner'].value_counts()
repeat_owners[repeat_owners > 1]

**Which repo is the oldest? Which is the newest?**

In [ ]:
oldest = data.loc[data['created_at'].idxmin()]
newest = data.loc[data['created_at'].idxmax()]

print("Oldest repo:", oldest['name'], "-", oldest['created_at'].date())
print("Newest repo:", newest['name'], "-", newest['created_at'].date())

**Which language has the highest *average* stars — not just the most repos?**

In [ ]:
avg_stars_by_lang = data.groupby('language')['stars'].mean().sort_values(ascending=False)
avg_stars_by_lang

## 6. Easy math (not new features — just one division)

**Star-to-fork ratio** — which repo do people star a lot but rarely copy?

In [ ]:
data['star_to_fork_ratio'] = data['stars'] / data['forks']
data[['name', 'stars', 'forks', 'star_to_fork_ratio']].sort_values('star_to_fork_ratio', ascending=False).head(5)

**Stars per day since it was created** — which repo is growing the fastest?

In [ ]:
data['stars_per_day'] = data['stars'] / data['repo_age_days']
data[['name', 'stars', 'repo_age_days', 'stars_per_day']].sort_values('stars_per_day', ascending=False).head(5)

## 7. Guess the winner 🎮
Run this a few times — guess which repo has more stars before it reveals the answer.

In [ ]:
def guess_game():
    r1, r2 = data.sample(2).to_dict('records')
    print("Which repo has more stars?")
    print(f"A) {r1['name']}")
    print(f"B) {r2['name']}")
    answer = input("Type A or B: ").strip().upper()

    winner = 'A' if r1['stars'] > r2['stars'] else 'B'
    print("Correct! \U0001F389" if answer == winner else "Not quite!")
    print(f"{r1['name']}: {r1['stars']} stars")
    print(f"{r2['name']}: {r2['stars']} stars")

guess_game()

## 8. Your GitHub Trends Report 📝

Fill this in after looking at your charts above:

1. The most popular repo is ______ because ______
2. The language with the most repos is ______
3. The fastest-growing repo (stars per day) is ______
4. Something that surprised me: ______